# Revolut FAQ RAG Chatbot

A minimal single-turn RAG chatbot over Revolut help articles.

**Stack:** `openai` for embeddings + chat, `numpy` for similarity search, `json` for loading. No frameworks, no vector DB — everything in memory.

In [1]:
%pip install -q openai numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Andrew\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import json
import os
import numpy as np
from openai import OpenAI


client = OpenAI(api_key="YOUR_API_KEY_HERE")

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-3.5-turbo" # feel free to replace with another WEAK model (use weak models to make it easier to find errors)
TOP_K = 4

## 1. Load articles

In [3]:
ARTICLES_PATH = "revolut_help_articles.jsonl"

articles = []
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        articles.append(json.loads(line))

print(f"Loaded {len(articles)} articles")
print("Example:", articles[0]["title"])

Loaded 786 articles
Example: How can I see my cashflow analytics?


## 2. Embed all articles

We embed `title + content_text` so the title contributes to retrieval. Batched to keep things fast.

In [4]:
def article_to_text(a):
    return f"{a['title']}\n\n{a['content_text']}"

def embed_texts(texts, batch_size=100):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return np.array(out, dtype=np.float32)

texts = [article_to_text(a) for a in articles]
embeddings = embed_texts(texts)

# L2-normalize once so cosine similarity is just a dot product
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (786, 1536)


## 3. Retrieval

In [5]:
def retrieve(query, k=TOP_K):
    q_emb = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    q_vec = np.array(q_emb, dtype=np.float32)
    q_vec = q_vec / np.linalg.norm(q_vec)

    scores = embeddings @ q_vec
    top_idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), articles[i]) for i in top_idx]

## 4. Single-turn chat

In [6]:
SYSTEM_PROMPT = (
    "You are a Revolut customer support assistant. "
    "Answer the user's question using ONLY the provided help articles. "
    "If the answer is not in the articles, say you don't know. "
    "Be concise and reference steps from the articles when relevant."
)

def format_context(hits):
    parts = []
    for rank, (idx, score, art) in enumerate(hits, start=1):
        parts.append(
            f"[Article {rank}] {art['title']}\n{art['content_text']}"
        )
    return "\n\n---\n\n".join(parts)

def ask(question, k=TOP_K):
    hits = retrieve(question, k=k)
    context = format_context(hits)

    user_msg = (
        f"Help articles:\n\n{context}\n\n"
        f"Question: {question}"
    )

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
    )
    answer = resp.choices[0].message.content
    return answer, hits

## 5. Try it

In [8]:
question = "как открыть аккаунт в монзо"

answer, hits = ask(question)

print("Q:", question)
print("\nA:", answer)
print("\nRetrieved articles:")
for rank, (idx, score, art) in enumerate(hits, start=1):
    print(f"  {rank}. [{score:.3f}] {art['title']}")

Q: как открыть аккаунт в монзо

A: I'm sorry, I don't have information on how to open an account with Monzo.

Retrieved articles:
  1. [0.369] Open a Revolut – Kids & Teens account
  2. [0.357] Duplicate account
  3. [0.337] Open an investment account
  4. [0.331] Change or verify email address
